[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iamjamesbowden/AG952/blob/main/assignments/march2026/AG952_Assignment_2026.ipynb)

# AG952 | Coursework Assignment 2026

**Textual Analytics for Accounting and Finance**  
University of Strathclyde Business School

---

This notebook serves three purposes: it allocates you to one of four research scenarios based on your student number; it provides the data loading and analytical infrastructure required to complete the analysis; and it records your methodological choices to a central log so that the module team can monitor approach diversity across the cohort. All analytical decisions and their justifications are your own. Your written report must explain and defend every choice made within this notebook, with explicit reference to the methodological literature.

Before you begin, save a personal copy of this notebook to your Google Drive (File > Save a copy in Drive) or to a personal GitHub repository. Do not modify the shared repository copy. Work only from your personal copy throughout the assignment.

**Estimated total time:** 8--12 hours across multiple sessions.  
**Release date:** 10 March 2026  
**Submission deadline:** 1 April 2026

In [ ]:
#@title Step 0 -- Clone the AG952 repository (run this first)

import os
import subprocess

REPO_URL = "https://github.com/iamjamesbowden/AG952.git"
REPO_DIR = "AG952"

if not os.path.exists(REPO_DIR):
    print("Cloning AG952 repository...")
    result = subprocess.run(["git", "clone", REPO_URL], capture_output=True, text=True)
    if result.returncode == 0:
        print("Repository cloned successfully.")
    else:
        print("Clone failed:")
        print(result.stderr)
else:
    print("Repository already present. Pulling latest changes...")
    result = subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"],
                            capture_output=True, text=True)
    print(result.stdout.strip() or "Already up to date.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
#@title Configuration -- set APPS_SCRIPT_URL before running

# Paste the deployed Apps Script web app URL here before running the notebook.
# Leave as an empty string if the endpoint has not yet been deployed.
APPS_SCRIPT_URL = "https://script.google.com/macros/s/AKfycbyrv8mSp2e2OpH4nqclZbmTNaRBO_xjMWoc1A8_0Y6bOwk6qfWlXnqH6HCtGpPigXDS/exec"

# Paths -- do not modify
# CORPUS_PATH is set dynamically in the Load Corpus cell after scenario allocation.
CORPUS_BASE_PATH  = "assignments/march2026/data"     # do not modify
LM_DICT_PATH      = "assignments/march2026/data/lm_dictionary.csv"
ALLOCATION_PATH   = "assignments/march2026/student_allocations.csv"
SEED_LABELS_PATH  = "assignments/march2026/nb_seed_labels.csv"

In [ ]:
#@title Step 1 -- Install and import dependencies

!pip install requests pandas numpy matplotlib seaborn scikit-learn nltk -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import json
import time
import warnings
import requests
import random

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import nltk
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)

warnings.filterwarnings("ignore")

print("Dependencies loaded successfully.")

## Scenario Allocation

Each student is assigned to one of four research scenarios based on their student number. The scenario determines which corpus and research question you will work with throughout the assignment. You must not change your allocated scenario.

In [ ]:
#@title Step 2 -- Enter your student number to receive your scenario allocation

SCENARIOS = {
    "A": {
        "id": "A",
        "title": "Climate and ESG Risk Language in the US Energy Sector (2019-2023)",
        "question": (
            "How has the linguistic framing of climate and ESG-related risk evolved "
            "across fossil fuel and renewable energy firms in their 10-K filings between "
            "2019 and 2023, and to what extent does language diverge systematically "
            "between firm types?"
        ),
        "corpus_note": (
            "100 10-K filings: 20 firms (oil/gas majors, integrated utilities, and "
            "pure-play renewables) x 5 annual filings (2019-2023)."
        ),
        "key_methods": [
            "LM dictionary sentiment (with attention to domain limitations for ESG vocabulary)",
            "Year-on-year cosine similarity to detect boilerplate versus substantive revision",
            "LDA topic modelling to discover latent climate risk themes across firm types"
        ]
    },
    "B": {
        "id": "B",
        "title": "Narrative Predictors of Corporate Financial Distress (2015-2023)",
        "question": (
            "Do the linguistic features of 10-K filings in the one to three years preceding "
            "a bankruptcy filing differ systematically from those of matched non-distressed "
            "peers, and can those features be used to train a reliable text-based classifier "
            "of financial distress?"
        ),
        "corpus_note": (
            "100 10-K filings: 50 distressed firms (last 10-K filed 2-3 years before "
            "bankruptcy) matched to 50 non-distressed peers by SIC code and asset quintile."
        ),
        "key_methods": [
            "LM uncertainty and litigious word lists as theoretically motivated features",
            "Gunning Fog Index as a measure of disclosure complexity in pre-distress filings",
            "Naive Bayes or logistic regression classifier with precision, recall, and F1 evaluation"
        ]
    },
    "C": {
        "id": "C",
        "title": "Risk Disclosure and the 2023 US Regional Banking Crisis",
        "question": (
            "Did the risk disclosures in 10-K filings from US regional banks in the period "
            "2020-2022 provide investors with meaningful warning of the vulnerabilities that "
            "materialised during the March 2023 banking crisis, and how does the language "
            "of subsequently failed banks compare to that of survivors?"
        ),
        "corpus_note": (
            "100 10-K filings: 25 US banks (3 failed, 5 stressed survivors, 10 unaffected "
            "regional banks, 7 large systemic banks) x 4 annual filings (2020-2022 plus "
            "final available filing)."
        ),
        "key_methods": [
            "LM uncertainty and litigious word lists applied to Risk Factors sections",
            "Year-on-year cosine similarity to identify inertial versus adaptive disclosure",
            "LDA topic modelling to detect shifts in interest rate and liquidity risk language"
        ]
    },
    "D": {
        "id": "D",
        "title": "Supply Chain Risk Disclosure Before and After COVID-19 (2019-2022)",
        "question": (
            "How did the language of supply chain risk disclosure in 10-K filings change "
            "across manufacturing and logistics-intensive firms in the two years before and "
            "the two years after the onset of the COVID-19 pandemic, and does the degree "
            "of linguistic change vary with firms' ex post supply chain vulnerability?"
        ),
        "corpus_note": (
            "100 10-K filings: 25 firms across automotive, consumer electronics, retail, "
            "industrial manufacturing, and logistics x 4 annual filings (FY2019-FY2022)."
        ),
        "key_methods": [
            "LM uncertainty word list, with explicit discussion of its supply chain vocabulary limitations",
            "Within-firm cosine similarity across years as a measure of disclosure change",
            "LDA topic modelling to identify a supply chain disruption topic and track its prevalence"
        ]
    }
}

# Modulo mapping for fallback allocation
MODULO_MAP = {0: "A", 1: "B", 2: "C", 3: "D"}


STUDENT_ID = input("Enter your student number: ").strip()


def allocate_scenario(student_id, allocation_path, modulo_map):
    """
    Look up student_id in the allocation CSV.
    Falls back to int(student_id) % 4 if the student ID is not found.
    Returns a scenario key: "A", "B", "C", or "D".
    """
    try:
        df = pd.read_csv(allocation_path, dtype=str)
        df["student_id"] = df["student_id"].str.strip()
        match = df[df["student_id"] == student_id.strip()]
        if not match.empty:
            return match.iloc[0]["scenario"].strip().upper(), "allocation file"
    except FileNotFoundError:
        print(f"Warning: allocation file not found at {allocation_path}.")
    except Exception as ex:
        print(f"Warning: could not read allocation file ({ex}).")

    # Fallback: modulo allocation
    try:
        numeric_id = int("".join(filter(str.isdigit, student_id)))
        scenario = modulo_map[numeric_id % 4]
        return scenario, "modulo fallback (student ID not found in allocation file)"
    except Exception:
        return "A", "default fallback (student ID could not be parsed)"


ASSIGNED_SCENARIO_KEY, ALLOCATION_METHOD = allocate_scenario(
    STUDENT_ID, ALLOCATION_PATH, MODULO_MAP
)
SCENARIO = SCENARIOS[ASSIGNED_SCENARIO_KEY]

print("\n" + "=" * 60)
print(f"  Student ID:  {STUDENT_ID}")
print(f"  Scenario:    {SCENARIO['id']} -- {SCENARIO['title']}")
print(f"  Allocated via: {ALLOCATION_METHOD}")
print("=" * 60)
print(f"\nResearch question:\n{SCENARIO['question']}")
print(f"\nCorpus:\n{SCENARIO['corpus_note']}")
print(f"\nKey methods to consider for this scenario:")
for m in SCENARIO["key_methods"]:
    print(f"  \u2022 {m}")
print("\nDo not change your allocated scenario. Record it in your report.")

In [ ]:
#@title Step 3 -- Load the corpus

# The corpus path is derived from your allocated scenario.
# Sections available in the corpus: item_1a (Risk Factors),
#   item_7 (MD&A), item_7a (Quantitative Disclosures About Market Risk).
# Filter to your chosen section with:
#   df = corpus[corpus["section"] == "item_1a"]

import os

CORPUS_PATH = os.path.join(
    CORPUS_BASE_PATH,
    f"scenario_{ASSIGNED_SCENARIO_KEY.lower()}",
    "corpus.csv"
)

# If FileNotFoundError is raised, confirm the repository is up to date:
# run `git pull origin main` in a terminal, then re-run this cell.
try:
    corpus = pd.read_csv(CORPUS_PATH)
    print(f"Corpus loaded from: {CORPUS_PATH}")
    print(f"Shape: {corpus.shape[0]} rows x {corpus.shape[1]} columns")
    print(f"Columns: {list(corpus.columns)}")
    print(f"\nUnique firms:    {corpus['firm'].nunique()}")
    print(f"Unique years:    {sorted(corpus['year'].unique())}")
    print(f"Sections:        {sorted(corpus['section'].unique())}")
    print(f"\nFirst 3 rows (transposed):")
    display(corpus.head(3).T)
except FileNotFoundError:
    print(f"ERROR: corpus not found at {CORPUS_PATH}.")
    print("Run `git pull origin main` to ensure the data files are present.")

## Pre-processing

You must implement a cleaning pipeline that prepares the raw 10-K text for downstream analysis and justify every decision in your report with explicit reference to Grimmer, Roberts and Stewart (2022). Five decisions are required, each with direct consequences for the validity of your results: superficial choices made without theoretical justification will be penalised in the marking scheme.

| Decision | Why it matters in financial text |
|---|---|
| Tokenisation method | Splits and hyphenated financial terms (e.g. *mark-to-market*) behave differently across tokenisers |
| Stop-word list | Standard lists remove modal verbs (*may*, *should*) and negations (*not*) that carry meaning in risk disclosures |
| Stemming vs. lemmatisation | Affects whether *liabilities* and *liability* are treated as the same token |
| Number handling | Dollar amounts, percentages, and CIK codes are near-unique tokens that inflate vocabulary size without contributing sentiment signal |
| TF-IDF weighting | Down-weighting frequent boilerplate terms may sharpen discriminative power but alters the interpretation of word counts |

In [ ]:
#@title Step 4 -- Record your pre-processing decisions

!pip install ipywidgets -q

import ipywidgets as widgets
from IPython.display import display

w_tokenisation = widgets.Dropdown(
    options=["word_tokenize (NLTK)", "split on whitespace", "spaCy tokeniser"],
    description="Tokenisation:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="500px")
)
w_stopwords = widgets.Dropdown(
    options=["Standard NLTK stopwords",
             "Finance-adjusted (modal verbs and negation retained)"],
    description="Stop-word list:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="500px")
)
w_normalisation = widgets.Dropdown(
    options=["Lemmatisation (WordNetLemmatizer)", "Stemming (PorterStemmer)", "None"],
    description="Normalisation:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="500px")
)
w_numbers = widgets.Dropdown(
    options=["Remove all numeric tokens", "Retain as-is",
             "Replace with placeholder <NUM>"],
    description="Number handling:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="500px")
)
w_tfidf = widgets.Dropdown(
    options=["Yes -- apply TF-IDF weighting", "No -- use raw term counts"],
    description="TF-IDF weighting:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="500px")
)

print("Select your pre-processing choices, then run the next cell to apply them.")
display(w_tokenisation, w_stopwords, w_normalisation, w_numbers, w_tfidf)

In [ ]:
#@title Step 5 -- Implement your pre-processing pipeline

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer


def preprocess(text: str) -> list:
    """
    Clean and tokenise a single document string.
    Wire your widget choices into each decision point below.

    Widget values available:
      w_tokenisation.value  -- controls tokeniser
      w_stopwords.value     -- controls stop-word list
      w_normalisation.value -- controls stemming / lemmatisation
      w_numbers.value       -- controls number handling
      w_tfidf.value         -- TF-IDF is applied at corpus level; see Step 8
    """
    # YOUR CODE HERE
    pass


# Test on the first row of the text column
sample_text = corpus["text"].iloc[0] if "text" in corpus.columns else ""
tokens = preprocess(sample_text)
print(f"First 30 tokens: {tokens[:30]}")

## Sentiment Analysis

You must apply one sentiment engine to the corpus and justify your choice in the report on at least three criteria: transparency of the underlying word list or model weights; domain specificity to financial and regulatory language; and computational requirements relative to the corpus size. Two approaches are scaffolded below. The Loughran-McDonald dictionary (Cell 6a) is interpretable and computationally lightweight but depends on the quality of your pre-processing choices. FinBERT (Cell 6b) is contextually richer but requires a GPU runtime and imposes a 512-token input limit that may affect long 10-K sections. Your report must engage with these trade-offs rather than simply reporting the approach you used.

In [ ]:
#@title Step 6 -- Select your sentiment model

w_sentiment = widgets.Dropdown(
    options=["lm_dictionary", "finbert"],
    description="Sentiment model:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="500px")
)
print("Select your sentiment model, then run the implementation cell below.")
display(w_sentiment)

In [ ]:
#@title Step 6a -- LM dictionary approach

try:
    lm_dict = pd.read_csv(LM_DICT_PATH)
    positive_words = set(lm_dict[lm_dict["Positive"] != 0]["Word"].str.lower())
    negative_words = set(lm_dict[lm_dict["Negative"] != 0]["Word"].str.lower())
    print(f"LM dictionary loaded: {len(positive_words)} positive, "
          f"{len(negative_words)} negative words.")
except FileNotFoundError:
    print(f"ERROR: LM dictionary not found at {LM_DICT_PATH}.")
    print("Run `git pull origin main` to ensure data files are present.")
    positive_words, negative_words = set(), set()


def lm_sentiment(text, positive_words, negative_words):
    """
    Compute Loughran-McDonald sentiment scores for a single document.

    Returns a dict with:
      pos_count      -- number of positive word matches
      neg_count      -- number of negative word matches
      net_sentiment  -- pos_count - neg_count
      sentiment_score -- (pos - neg) / (pos + neg), or np.nan if denominator == 0
    """
    tokens = [t.lower() for t in text.split()]
    pos = sum(1 for t in tokens if t in positive_words)
    neg = sum(1 for t in tokens if t in negative_words)
    denom = pos + neg
    score = (pos - neg) / denom if denom > 0 else np.nan
    return {"pos_count": pos, "neg_count": neg,
            "net_sentiment": pos - neg, "sentiment_score": score}


if "text" in corpus.columns and positive_words:
    lm_results = corpus["text"].apply(
        lambda t: lm_sentiment(t, positive_words, negative_words)
    )
    corpus["lm_pos"]   = lm_results.apply(lambda d: d["pos_count"])
    corpus["lm_neg"]   = lm_results.apply(lambda d: d["neg_count"])
    corpus["lm_net"]   = lm_results.apply(lambda d: d["net_sentiment"])
    corpus["lm_score"] = lm_results.apply(lambda d: d["sentiment_score"])
    print("LM sentiment scores computed.")
    display(corpus["lm_score"].describe().to_frame())

In [ ]:
#@title Step 6b -- FinBERT approach (GPU runtime required)

# Before running this cell, switch to a GPU runtime:
# Runtime > Change runtime type > T4 GPU
#
# !pip install transformers torch -q
#
# from transformers import pipeline
# finbert = pipeline("text-classification",
#                    model="ProsusAI/finbert",
#                    tokenizer="ProsusAI/finbert",
#                    device=0)  # device=0 for GPU
#
# FinBERT accepts a maximum of 512 tokens per input.
# For 10-K sections exceeding this limit, split into
# overlapping chunks of 400 tokens and aggregate scores.
# For corpora exceeding 500 documents, use batched inference:
# finbert(texts, batch_size=32, truncation=True, max_length=512)

# YOUR CODE HERE
pass

## Secondary Metric

You must supplement your sentiment analysis with one secondary metric and explain in your report what incremental information it provides that sentiment alone cannot. Three options are scaffolded: the Gunning Fog readability index (Li, 2008), which captures disclosure complexity and has been linked to earnings persistence; year-on-year cosine similarity (Brown and Tucker, 2011), which distinguishes substantive revision from boilerplate repetition; and Latent Dirichlet Allocation (Grimmer et al., 2022, Ch. 13), which identifies latent thematic structure without requiring a predefined vocabulary. Your choice should be theoretically motivated by your scenario's research question.

In [ ]:
#@title Step 7 -- Select your secondary metric

w_secondary = widgets.Dropdown(
    options=["fog_index", "cosine_similarity", "lda"],
    description="Secondary metric:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="500px")
)
print("Select your secondary metric, then implement it in the scaffold below.")
display(w_secondary)

In [ ]:
#@title Step 8 -- Implement your secondary metric


def gunning_fog(text: str) -> float:
    """
    Gunning Fog Index.
    Formula: 0.4 * (avg_sentence_length + pct_complex_words)
    Complex words: words with 3 or more syllables.
    Reference: Li (2008), Journal of Accounting and Economics.
    """
    # YOUR CODE HERE
    pass


def cosine_year_on_year(text_year_t: str, text_year_t1: str) -> float:
    """
    TF-IDF cosine similarity between two texts from consecutive years.
    Returns a float in [0, 1]; values near 1 indicate minimal revision.
    Reference: Brown and Tucker (2011), Journal of Accounting Research.
    """
    # YOUR CODE HERE
    pass


def run_lda(texts: list, n_topics: int = 10) -> tuple:
    """
    Fit an LDA model to pre-processed texts.
    Requires gensim: !pip install gensim -q
    Returns: (lda_model, topic_word_matrix, document_topic_matrix)
    Reference: Grimmer et al. (2022), Text as Data, Ch. 13.
    """
    # YOUR CODE HERE
    pass

## Visualisation

You must produce a minimum of two professional-standard visualisations and include them in your report. Each figure must have clearly labelled axes, a descriptive title, and a legend where multiple series are shown. Save each figure to disk before calling `plt.show()` using `plt.savefig('figN.png', dpi=150, bbox_inches='tight')` so that it persists across sessions and can be incorporated directly into your written submission.

In [ ]:
#@title Step 9 -- Produce visualisations

# YOUR CODE HERE
#
# Reminder: call plt.savefig() before plt.show() so the file is written
# to disk before the figure is cleared from memory.
# Example:
#   plt.savefig("fig1.png", dpi=150, bbox_inches="tight")
#   plt.show()
pass

## Manual Validation

Automated sentiment methods require empirical validation. You must draw a random sample of 100 sentences from the corpus, classify each sentence as positive, negative, or neutral by reading it in context, and compare your human judgements to the automated scores. Quantify the machine-human gap using a metric of your choice (e.g. percentage agreement, Cohen's kappa) and discuss where failures are concentrated: hedged language (*it is possible that...*), negation (*did not improve*), and context-dependent terms (*risk* as threat versus *risk management* as competence) are the most common failure modes in financial text and should receive explicit attention in your report.

In [ ]:
#@title Step 10 -- Draw random sample for manual validation

random.seed(42)

if "text" in corpus.columns:
    all_sentences = []
    for _, row in corpus.iterrows():
        for sent in str(row["text"]).split("."):
            sent = sent.strip()
            if len(sent.split()) >= 5:
                all_sentences.append({
                    "firm": row.get("firm", ""),
                    "year": row.get("year", ""),
                    "sentence_text": sent
                })

    sample = random.sample(all_sentences, min(100, len(all_sentences)))
    validation_df = pd.DataFrame(sample)
    validation_df.insert(0, "sentence_id", range(1, len(validation_df) + 1))
    validation_df["human_label"]     = ""
    validation_df["automated_label"] = ""

    validation_df.to_csv("manual_validation_sample.csv", index=False)
    print("manual_validation_sample.csv written.")
    print(f"Rows: {len(validation_df)}")
    display(validation_df.head())
else:
    print("No 'text' column found in corpus. Load the corpus first (Step 3).")

In [ ]:
#@title Step 11 -- Load Naive Bayes seed labels

# These labels are provided as training data for the optional supervised
# extension in Step 12. They must not be used to shortcut the manual
# validation exercise in Step 10.
try:
    seed_labels = pd.read_csv(SEED_LABELS_PATH)
    print(f"Seed labels loaded: {seed_labels.shape[0]} rows x {seed_labels.shape[1]} columns")
    print(f"Columns: {list(seed_labels.columns)}")
    print(f"\nLabel distribution:")
    display(seed_labels["label"].value_counts().to_frame())
except FileNotFoundError:
    print(f"WARNING: seed labels file not found at {SEED_LABELS_PATH}.")
    print("Run `git pull origin main` to ensure the file is present.")

In [ ]:
#@title Step 12 -- Naive Bayes classifier (optional extension)

w_naive_bayes = widgets.Dropdown(
    options=["no", "yes"],
    description="Naive Bayes:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="500px")
)
print("Are you implementing the optional Naive Bayes extension?")
display(w_naive_bayes)

# YOUR CODE HERE
#
# Suggested skeleton:
#   vectorizer = TfidfVectorizer(max_features=5000)
#   X = vectorizer.fit_transform(seed_labels["sentence"])
#   y = seed_labels["label"]
#   X_train, X_test, y_train, y_test = train_test_split(
#       X, y, test_size=0.2, random_state=42
#   )
#   clf = MultinomialNB()
#   clf.fit(X_train, y_train)
#   y_pred = clf.predict(X_test)
#   print(classification_report(y_test, y_pred))
#   print(confusion_matrix(y_test, y_pred))

In [ ]:
#@title Step 13 -- Submit your methodological choices

# Run this cell after finalising all choices above.
# You may re-run it if you change a choice; each submission is
# recorded separately and the submission_count column will increment.


def submit_choices(apps_script_url, student_id, scenario_id):
    if not apps_script_url:
        print("APPS_SCRIPT_URL is not set. Choices not submitted.")
        print("Contact your module leader if you are unsure how to proceed.")
        return

    payload = {
        "student_id":           student_id,
        "scenario":             scenario_id,
        "tokenisation_method":  w_tokenisation.value,
        "stopword_list":        w_stopwords.value,
        "normalisation_method": w_normalisation.value,
        "number_handling":      w_numbers.value,
        "tfidf_weighting":      w_tfidf.value,
        "sentiment_model":      w_sentiment.value,
        "secondary_metric":     w_secondary.value,
        "naive_bayes":          w_naive_bayes.value,
    }

    try:
        response = requests.post(
            apps_script_url,
            data=json.dumps(payload),
            headers={"Content-Type": "application/json"},
            timeout=15
        )
        result = response.json()
        if result.get("status") == "success":
            print(f"Choices submitted. This is submission number "
                  f"{result.get('submission', '?')} "
                  f"for student ID {student_id}.")
        else:
            print(f"Submission returned an error: "
                  f"{result.get('message', 'unknown error')}")
    except requests.exceptions.Timeout:
        print("The request timed out. Check your internet connection and try again.")
    except Exception as ex:
        print(f"Submission failed: {ex}")


submit_choices(APPS_SCRIPT_URL, STUDENT_ID, ASSIGNED_SCENARIO_KEY)

## Critical Evaluation

The final section of your report must critically evaluate the methodology you have implemented: where the automated analysis is likely to be reliable, where it is not, and what modifications would improve robustness. You should engage directly with the validation framework set out in Grimmer, Roberts and Stewart (2022), distinguishing between measurement validity (does the method capture the construct of interest?), convergent validity (does it agree with human judgement?), and discriminant validity (does it detect differences that exist in the underlying text?). You must also engage with at least one empirical paper that assesses the limitations of the specific method you have chosen, rather than relying solely on theoretical arguments.

## References

Brown, S.V. and Tucker, J.W. (2011) 'Large-sample evidence on firms' year-over-year MD&A modifications', *Journal of Accounting Research*, 49(2), pp. 309--346.

Grimmer, J., Roberts, M.E. and Stewart, B.M. (2022) *Text as Data: A New Framework for Machine Learning and the Social Sciences*. Princeton University Press.

Li, F. (2008) 'Annual report readability, current earnings, and earnings persistence', *Journal of Accounting and Economics*, 45(2--3), pp. 221--247.

Loughran, T. and McDonald, B. (2011) 'When is a liability not a liability? Textual analysis, dictionaries, and 10-Ks', *Journal of Finance*, 66(1), pp. 35--65.